# SQLCoder 7B — Fine-Tuning Pipeline
Run cells one by one. Paste output in chat after each step.

In [1]:
import sys
!{sys.executable} -m pip install -q pandas scikit-learn transformers accelerate peft datasets trl bitsandbytes plotly loguru rich tqdm
print('✅ Done')

✅ Done


In [2]:
import os, json, re, csv, time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

print(f'Device  : {DEVICE.upper()}')
print(f'PyTorch : {torch.__version__}')

Device  : MPS
PyTorch : 2.11.0


## Step 1 — Load & Inspect CSV
Put your CSV in the same folder as this notebook.

In [3]:
CSV_PATH = 'text-to-sql-100k(in).csv'   # change if your filename differs

df = pd.read_csv(CSV_PATH)
print(f'Rows    : {len(df):,}')
print(f'Columns : {list(df.columns)}')
print(f'Nulls   :\n{df.isnull().sum()}')
print()
print('SQL TYPE DISTRIBUTION:')
for t, n in df['sql_type'].value_counts().items():
    bar = '█' * min(n // max(len(df)//50, 1), 50)
    print(f'  {t:<14} {n:>6,}  {bar}')
df.head(3)

Rows    : 100,000
Columns : ['structured_input', 'sql', 'sql_type']
Nulls   :
structured_input    0
sql                 0
sql_type            0
dtype: int64

SQL TYPE DISTRIBUTION:
  SELECT         89,484  ████████████████████████████████████████████
  INSERT          3,257  █
  DELETE          3,240  █
  UPDATE          2,850  █
  UNKNOWN         1,169  


,structured_input,sql,sql_type
0,prompt: What is the total volume of timber sol...,"SELECT salesperson_id, name, SUM(volume) as to...",SELECT
1,prompt: List all the unique equipment types an...,"SELECT equipment_type, SUM(maintenance_frequen...",SELECT
2,prompt: How many marine species are found in t...,SELECT COUNT(*) FROM marine_species WHERE loca...,SELECT


## Step 2 — Clean & Split

In [4]:
required = ['structured_input', 'sql', 'sql_type']
missing  = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

before = len(df)
df = df.dropna(subset=required)
df['structured_input'] = df['structured_input'].astype(str).str.strip()
df['sql']              = df['sql'].astype(str).str.strip()
df['sql_type']         = df['sql_type'].astype(str).str.strip().str.upper()
df = df[(df['structured_input'].str.len() > 0) & (df['sql'].str.len() > 0)].reset_index(drop=True)
print(f'Cleaned: {before:,} → {len(df):,} rows (dropped {before-len(df)})')

# merge rare types
rare = df['sql_type'].value_counts()
rare = rare[rare < 3].index.tolist()
if rare:
    print(f'Rare types → OTHER: {rare}')
    df.loc[df['sql_type'].isin(rare), 'sql_type'] = 'OTHER'

def safe_split(data, test_size, seed, label=''):
    try:
        return train_test_split(data, test_size=test_size, stratify=data['sql_type'], random_state=seed)
    except ValueError:
        print(f'  ⚠ {label}: falling back to random split')
        return train_test_split(data, test_size=test_size, random_state=seed)

train_df, temp_df = safe_split(df, 0.20, 42, 'train/temp')
val_df,   test_df = safe_split(temp_df, 0.50, 42, 'val/test')

print(f'Train : {len(train_df):>6,} ({len(train_df)/len(df):.0%})')
print(f'Val   : {len(val_df):>6,} ({len(val_df)/len(df):.0%})')
print(f'Test  : {len(test_df):>6,} ({len(test_df)/len(df):.0%})')
print()
print(f"  {'TYPE':<14} {'TRAIN':>7} {'VAL':>7} {'TEST':>7}")
print('  ' + '-'*38)
for t in sorted(df['sql_type'].unique()):
    print(f"  {t:<14} {(train_df['sql_type']==t).sum():>7,} {(val_df['sql_type']==t).sum():>7,} {(test_df['sql_type']==t).sum():>7,}")

Cleaned: 100,000 → 100,000 rows (dropped 0)
Train : 80,000 (80%)
Val   : 10,000 (10%)
Test  : 10,000 (10%)

  TYPE             TRAIN     VAL    TEST
  --------------------------------------
  DELETE           2,592     324     324
  INSERT           2,606     325     326
  SELECT          71,587   8,949   8,948
  UNKNOWN            935     117     117
  UPDATE           2,280     285     285


## Step 3 — Write JSONL Files

In [5]:
TRAIN_TMPL = '### Task\nGenerate a SQL query to answer the following question.\n\n### Input\n{structured_input}\n\n### Response\n{sql}'
INFER_TMPL = '### Task\nGenerate a SQL query to answer the following question.\n\n### Input\n{structured_input}\n\n### Response\n'

def to_record(row, infer=False):
    tmpl = INFER_TMPL if infer else TRAIN_TMPL
    return {'text': tmpl.format(**row), 'sql': row['sql'], 'sql_type': row['sql_type'], 'structured_input': row['structured_input']}

def write_jsonl(records, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in records: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'  Wrote {len(records):>6,} → {path}')

write_jsonl([to_record(r)           for r in train_df.to_dict('records')], 'data/train.jsonl')
write_jsonl([to_record(r)           for r in val_df.to_dict('records')],   'data/val.jsonl')
write_jsonl([to_record(r)           for r in test_df.to_dict('records')],  'data/test.jsonl')
write_jsonl([to_record(r, infer=True) for r in test_df.to_dict('records')],'data/test_inference.jsonl')
print()
print('Sample prompt:')
print('-'*60)
print(to_record(train_df.iloc[0].to_dict())['text'])
print('-'*60)
print('✅ Step 1 done — paste output in chat')

  Wrote 80,000 → data/train.jsonl
  Wrote 10,000 → data/val.jsonl
  Wrote 10,000 → data/test.jsonl
  Wrote 10,000 → data/test_inference.jsonl

Sample prompt:
------------------------------------------------------------
### Task
Generate a SQL query to answer the following question.

### Input
prompt: Delete records of biosensors with sensitivity lower than 0.95 from the biosensors table schema: CREATE TABLE biosensors (id INT, name VARCHAR(50), type VARCHAR(50), sensitivity FLOAT, specificity FLOAT, company_name VARCHAR(50)); INSERT INTO biosensors (id, name, type, sensitivity, specificity, company_name) VALUES (1, 'BioGlucose', 'Glucose', 0.95, 0.98, 'BioCorp'), (2, 'BioOxygen', 'Oxygen', 0.92, 0.96, 'BioCorp'), (3, 'BioPressure', 'Pressure', 0.98, 0.99, 'BioCorp');

### Response
DELETE FROM biosensors WHERE sensitivity < 0.95;
------------------------------------------------------------
✅ Step 1 done — paste output in chat


  Wrote 80,000 → data/train.jsonl
  Wrote 10,000 → data/val.jsonl
  Wrote 10,000 → data/test.jsonl
  Wrote 10,000 → data/test_inference.jsonl

Sample prompt:
------------------------------------------------------------
### Task
Generate a SQL query to answer the following question.

### Input
prompt: Delete records of biosensors with sensitivity lower than 0.95 from the biosensors table schema: CREATE TABLE biosensors (id INT, name VARCHAR(50), type VARCHAR(50), sensitivity FLOAT, specificity FLOAT, company_name VARCHAR(50)); INSERT INTO biosensors (id, name, type, sensitivity, specificity, company_name) VALUES (1, 'BioGlucose', 'Glucose', 0.95, 0.98, 'BioCorp'), (2, 'BioOxygen', 'Oxygen', 0.92, 0.96, 'BioCorp'), (3, 'BioPressure', 'Pressure', 0.98, 0.99, 'BioCorp');

### Response
DELETE FROM biosensors WHERE sensitivity < 0.95;
------------------------------------------------------------
✅ Step 1 done — paste output in chatthe 